In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import linregress
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import pandas as pd

nav = pd.read_csv("../data/raw/02_nav_history.csv")
benchmark = pd.read_csv("../data/raw/10_benchmark_indices.csv")

In [ ]:
nav["date"] = pd.to_datetime(nav["date"])


In [ ]:
nav = nav.sort_values(["amfi_code", "date"])

In [ ]:
nav["daily_return"] = nav.groupby("amfi_code")["nav"].pct_change()

In [ ]:
nav.head()

In [ ]:
nav.columns

In [ ]:
benchmark["index_name"].unique()

In [ ]:
benchmark["index_name"].value_counts()

In [ ]:
benchmark["date"] = pd.to_datetime(benchmark["date"])

In [ ]:
nifty100 = benchmark[
    benchmark["index_name"] == "NIFTY100"
].copy()

In [ ]:
nifty100.head()

In [ ]:
nifty100 = nifty100.sort_values("date")

In [ ]:
nifty100["bench_return"] = nifty100["close_value"].pct_change()


In [ ]:
nifty100.head()

In [ ]:
nav = nav.merge(
    nifty100[["date", "bench_return"]],
    on="date",
    how="left"
)

In [ ]:
nav.head()

In [ ]:
nav.to_csv("../data/processed/returns_completed.csv", index=False)

In [ ]:
def calculate_cagr(df, years):
    """
    Calculate Compound Annual Growth Rate (CAGR)
    using historical NAV values.

    Parameters:
        df (DataFrame): Fund NAV data
        years (int): Investment period in years

    Returns:
        float: CAGR value
    """
    start_nav = df["nav"].iloc[0]
    end_nav = df["nav"].iloc[-1]

    return (end_nav / start_nav) ** (1 / years) - 1
cagr_results = []

for fund, group in nav.groupby("amfi_code"):

    group = group.sort_values("date")

    result = {
        "amfi_code": fund
    }

    # 1 Year CAGR
    if len(group) >= 252:
        result["cagr_1y"] = calculate_cagr(group.tail(252), 1)

    # 3 Year CAGR
    if len(group) >= 756:
        result["cagr_3y"] = calculate_cagr(group.tail(756), 3)

    # Add result to list
    cagr_results.append(result)

# Create DataFrame
cagr_df = pd.DataFrame(cagr_results)

# View results
cagr_df.head()

In [ ]:
cagr_df.to_csv("../data/processed/cagr_report.csv", index=False)


In [ ]:
RF = 0.065

In [ ]:
def sharpe_ratio(returns):
    """
    Calculate annualized Sharpe Ratio
    using daily fund returns.

    Parameters:
        returns (Series): Daily returns

    Returns:
        float: Sharpe Ratio
    """
    returns = returns.dropna()

    excess_returns = returns - (RF / 252)

    return (
        excess_returns.mean()
        / returns.std()
    ) * np.sqrt(252)

In [ ]:
sharpe_df = (
    nav.groupby("amfi_code")["daily_return"]
    .apply(sharpe_ratio)
    .reset_index(name="sharpe_ratio")
)

In [ ]:
sharpe_df.sort_values(
    "sharpe_ratio",
    ascending=False
).head()

In [ ]:
sharpe_df.to_csv("../data/processed/sharpe_values.csv", index=False)


In [ ]:
def sortino_ratio(returns):
    """
    Calculate Sortino Ratio using
    downside volatility.

    Parameters:
        returns (Series): Daily returns

    Returns:
        float: Sortino Ratio
    """
    returns = returns.dropna()

    excess_returns = returns - (RF / 252)

    downside_returns = returns[returns < 0]

    downside_std = downside_returns.std()

    return (
        excess_returns.mean()
        / downside_std
    ) * np.sqrt(252)

In [ ]:
sortino_df = (
    nav.groupby("amfi_code")["daily_return"]
    .apply(sortino_ratio)
    .reset_index(name="sortino_ratio")
)

In [ ]:
sortino_df.head()

In [ ]:
sortino_df.to_csv("../data/processed/sortino_values.csv", index=False)


In [ ]:
from scipy.stats import linregress
import numpy as np

In [ ]:
def calculate_alpha_beta(df):
    """
    Calculate Alpha and Beta by comparing
    fund returns with benchmark returns.

    Parameters:
        df (DataFrame): Fund and benchmark returns

    Returns:
        Series: Alpha and Beta values
    """

    df = df.dropna(
        subset=["daily_return", "bench_return"]
    )

    if len(df) < 30:
        return pd.Series({
            "alpha": np.nan,
            "beta": np.nan
        })

    beta, alpha, r_value, p_value, std_err = linregress(
        df["bench_return"],
        df["daily_return"]
    )

    return pd.Series({
        "alpha": alpha * 252,
        "beta": beta
    })

In [ ]:
aalpha_beta_df = (
    nav.groupby("amfi_code")[["daily_return", "bench_return"]]
       .apply(calculate_alpha_beta)
       .reset_index()
)

In [ ]:
alpha_beta_df.head()

In [ ]:
alpha_beta_df.to_csv(
    "../data/processed/alpha_beta.csv",
    index=False
)

In [ ]:
def max_drawdown(nav_series):
     """
    Calculate maximum drawdown from
    historical NAV values.

    Parameters:
        nav_series (Series): NAV values

    Returns:
        float: Maximum Drawdown
    """
    running_max = nav_series.cummax()

    drawdown = (
        nav_series / running_max
    ) - 1

    return drawdown.min()

In [ ]:
mdd_df = (
    nav.groupby("amfi_code")["nav"]
       .apply(max_drawdown)
       .reset_index(name="max_drawdown")
)

In [ ]:
mdd_df.head()

In [ ]:
mdd_df.to_csv(
    "../data/processed/max_drawdown.csv",
    index=False
)

In [ ]:
import numpy as np
import pandas as pd

tracking_errors = []

for fund in top5_funds:

    temp = nav[nav["amfi_code"] == fund].copy()

    temp = temp.dropna(
        subset=["daily_return", "bench_return"]
    )

    tracking_error = (
        (temp["daily_return"] - temp["bench_return"])
        .std()
        * np.sqrt(252)
    )

    tracking_errors.append({
        "amfi_code": fund,
        "tracking_error": tracking_error
    })

tracking_error_df = pd.DataFrame(tracking_errors)

tracking_error_df

In [ ]:
tracking_error_df.to_csv(
    "../data/processed/tracking_error.csv",
    index=False
)

In [ ]:
# Expense Ratio
expense_df = fund_master[
    ["amfi_code", "expense_ratio_pct"]
].copy()

# Merge all metrics
scorecard_df = (
    cagr_df
    .merge(sharpe_df, on="amfi_code")
    .merge(sortino_df, on="amfi_code")
    .merge(alpha_beta_df, on="amfi_code")
    .merge(mdd_df, on="amfi_code")
    .merge(expense_df, on="amfi_code")
)

# Create ranks
scorecard_df["return_rank"] = (
    scorecard_df["cagr_3y"]
    .rank(ascending=False)
)

scorecard_df["sharpe_rank"] = (
    scorecard_df["sharpe_ratio"]
    .rank(ascending=False)
)

scorecard_df["alpha_rank"] = (
    scorecard_df["alpha"]
    .rank(ascending=False)
)

# Lower expense ratio is better
scorecard_df["expense_rank"] = (
    scorecard_df["expense_ratio_pct"]
    .rank(ascending=True)
)

# Lower drawdown magnitude is better
scorecard_df["drawdown_rank"] = (
    scorecard_df["max_drawdown"]
    .rank(ascending=False)
)

# Composite Score
scorecard_df["fund_score"] = (
      scorecard_df["return_rank"] * 0.30
    + scorecard_df["sharpe_rank"] * 0.25
    + scorecard_df["alpha_rank"] * 0.20
    + scorecard_df["expense_rank"] * 0.15
    + scorecard_df["drawdown_rank"] * 0.10
)

# Convert to 0-100 scale
scorecard_df["fund_score_100"] = (
    (
        scorecard_df["fund_score"].max()
        - scorecard_df["fund_score"]
    )
    /
    (
        scorecard_df["fund_score"].max()
        - scorecard_df["fund_score"].min()
    )
) * 100

# Best funds first
scorecard_df = scorecard_df.sort_values(
    "fund_score_100",
    ascending=False
)

# View Top 10 Funds
scorecard_df.head(10)

In [ ]:
scorecard_df.to_csv(
    "../data/processed/fund_scorecard.csv",
    index=False
)

In [ ]:
top5_funds = scorecard_df.head(5)["amfi_code"].tolist()

top5_funds

In [ ]:
top5_nav = nav[
    nav["amfi_code"].isin(top5_funds)
].copy()

In [ ]:
nifty50 = benchmark[
    benchmark["index_name"] == "NIFTY50"
].copy()

nifty50["date"] = pd.to_datetime(
    nifty50["date"]
)

nifty50 = nifty50.sort_values("date")

In [ ]:
top5_nav["normalized"] = (
    top5_nav.groupby("amfi_code")["nav"]
    .transform(lambda x: x / x.iloc[0] * 100)
)

nifty50["normalized"] = (
    nifty50["close_value"]
    / nifty50["close_value"].iloc[0]
    * 100
)

nifty100["normalized"] = (
    nifty100["close_value"]
    / nifty100["close_value"].iloc[0]
    * 100
)

In [ ]:
import matplotlib.pyplot as plt

fund_name_map = (
    fund_master[
        ["amfi_code", "scheme_name"]
    ]
    .set_index("amfi_code")["scheme_name"]
    .to_dict()
)

plt.figure(figsize=(14,8))

for fund in top5_funds:

    fund_data = top5_nav[
        top5_nav["amfi_code"] == fund
    ]

    plt.plot(
        fund_data["date"],
        fund_data["normalized"],
        label=fund_name_map.get(fund, str(fund))
    )

plt.plot(
    nifty50["date"],
    nifty50["normalized"],
    label="NIFTY50",
    linewidth=3
)

plt.plot(
    nifty100["date"],
    nifty100["normalized"],
    label="NIFTY100",
    linewidth=3
)

plt.title("Top 5 Funds vs NIFTY50 & NIFTY100")
plt.xlabel("Date")
plt.ylabel("Growth Index (Base = 100)")
plt.legend()
plt.grid(True)

plt.savefig(
    "../charts/benchmark_comparison.png",
    bbox_inches="tight"
)

plt.show()

In [ ]:
print("NIFTY50 Start:", nifty50["close_value"].iloc[0])
print("NIFTY50 End:", nifty50["close_value"].iloc[-1])

print("NIFTY100 Start:", nifty100["close_value"].iloc[0])
print("NIFTY100 End:", nifty100["close_value"].iloc[-1])

In [ ]:
import os

os.listdir("../data/processed")